# Reworking DnD classes

> Import the character sheet fillable pdf and use a python library to fill in the data the user creates through localhost

First, let's handle imports.

In [ ]:
from enum import Enum
from pydantic import BaseModel, Field, computed_field, model_validator
from typing import Annotated, Any, Dict, List, Literal, Tuple, Union

Now we can define our static rules through Enums.

In [50]:
class Ability(str, Enum):
	STR = "Strength"
	DEX = "Dexterity"
	CON = "Constitution"
	INT = "Intelligence"
	WIS = "Wisdom"
	CHA = "Charisma"


class Size(str, Enum):
	SMALL = "Small"
	MEDIUM = "Medium"

In [51]:
class AbilityScores(BaseModel):
	strength: int = Field(ge=1, le=30, default=10)
	dexterity: int = Field(ge=1, le=30, default=10)
	constitution: int = Field(ge=1, le=30, default=10)
	intelligence: int = Field(ge=1, le=30, default=10)
	wisdom: int = Field(ge=1, le=30, default=10)
	charisma: int = Field(ge=1, le=30, default=10)

Let's make a base model for races now and then get into classes.

In [52]:
class BaseRace(BaseModel):
	size: Size = Size.MEDIUM
	speed: int = 30
	languages: List[str] = ["Common"]

In [ ]:
class Dragonborn(BaseRace):
	race_type: Literal["Dragonborn"] = "Dragonborn"
	draconic_ancestry: Literal[
		"Black",
		"Blue",
		"Brass",
		"Bronze",
		"Copper",
		"Gold",
		"Green",
		"Red",
		"Silver",
		"White",
	]

	@computed_field
	@property
	def damage_resistance_type(self) -> str:
		return {
			"Black": "Acid",
			"Blue": "Lightning",
			"Brass": "Fire",
			"Bronze": "Lightning",
			"Copper": "Acid",
			"Gold": "Fire",
			"Green": "Poison",
			"Red": "Fire",
			"Silver": "Cold",
			"White": "Cold",
		}.get(self.draconic_ancestry, "Black")
	
	@computed_field
	@property
	def breath_weapon(self) -> Union[int, Tuple[int, int]]:
		return {
			"Black": (5, 30),
			"Blue": (5, 30),
			"Brass": (5, 30),
			"Bronze": (5, 30),
			"Copper": (5, 30),
			"Gold": 15,
			"Green": 15,
			"Red": 15,
			"Silver": 15,
			"White": 15,
		}.get(self.draconic_ancestry, "Black")


class Dwarf(BaseRace):
	race_type: Literal["Dwarf"] = "Dwarf"
	darkvision_radius: int = 120


class Elf(BaseRace):
	race_type: Literal["Elf"] = "Elf"
	lineage: Literal["Drow", "High Elf", "Wood Elf"] = "Drow"
	darkvision_radius: int = 60
	fey_ancestry: bool = True
	keen_senses: List[str] = ["Insight", "Perception", "Survival"]
	trance: bool = True

In [54]:
DnDRace = Annotated[Union[Dwarf, Elf, Dragonborn], Field(discriminator="race_type")]

Now let's go over base classes.

In [ ]:
class BaseClassLevel(BaseModel):
	level: int = Field(ge=1, le=20, default=1)
	subclass: str | None = None

In [ ]:
class Fighter(BaseClassLevel):
	class_type: Literal["Fighter"] = "Fighter"
	fighting_style: str

	@computed_field
	def action_surges(self) -> int:
		if self.level >= 17:
			return 2
		if self.level >= 2:
			return 1
		return 0


class Wizard(BaseClassLevel):
	class_type: Literal["Wizard"] = "Wizard"
	spellbook: List[str] = Field(default_factory=list)
	prepared_spells: List[str] = Field(default_factory=list)

In [57]:
DnDClass = Annotated[Union[Fighter, Wizard], Field(discriminator="class_type")]

Here's our character model.

In [ ]:
class Character(BaseModel):
	name: str
	base_stats: AbilityScores
	race: DnDRace
	classes: List[DnDClass] = Field(default_factory=list)

	@model_validator(mode="after")
	def validate_total_level(self) -> "Character":
		current_total = sum(c.level for c in self.classes)
		if current_total > 20:
			raise ValueError(
				f"Total character cannot exceed 20. Current: {current_total}"
			)
		return self

	@computed_field
	@property
	def total_level(self) -> int:
		return sum(c.level for c in self.classes)

	@computed_field
	@property
	def proficiency_bonus(self) -> int:
		level = self.total_level
		return ((level - 1) // 4) + 2

	@computed_field
	def character_classes_summary(self) -> str:
		return " / ".join(f"{c.class_type} {c.level}" for c in self.classes)

Let's look at some example usage.

In [ ]:
legolas_data: Dict[str, Any] = {
	"name": "Legolas",
	"base_stats": {
		"strength": 8,
		"dexterity": 16,
		"constitution": 12,
		"intelligence": 18,
		"wisdom": 10,
		"charisma": 14,
	},
	"race": {
		"race_type": "Elf",
		"darkvision_radius": 60,
		"keen_senses": ["Insight", "Perception", "Survival"],
		"languages": ["Common", "Elvish"],
	},
	"classes": [
		{
			"class_type": "Fighter",
			"level": 2,
			"fighting_style": "Defense",
		},
		{
			"class_type": "Wizard",
			"level": 3,
			"subclass": "School of Evocation",
			"spellbook": ["Mage Armor", "Fireball", "Shield"],
		},
	],
}

In [ ]:
legolas = Character.model_validate(legolas_data)

print(f"Name: {legolas.name}")
print(f"Class: {legolas.character_classes_summary}")
print(f"Proficiency Bonus: +{legolas.proficiency_bonus}")
if isinstance(legolas.classes[0], Fighter):
	print(f"Action Surges Available: {legolas.classes[0].action_surges}")
else:
	print("This class doesn't have Action Surges")

Name: Legolas
Class: Fighter 2 / Wizard 3
Proficiency Bonus: +3
Action Surges Available: 1
